### 1. 数据读取

In [4]:
!pip install pandas "dask[complete]" numpy pyarrow psutil

In [5]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import gc
import psutil
import os
import warnings
warnings.filterwarnings('ignore')

In [9]:
filepath = 'user_behavior_processed.csv'

file_size_gb = os.path.getsize(filepath) / 1024**3
print(f"文件大小: {file_size_gb:.2f} GB")

文件大小: 0.46 GB


In [19]:
df = pd.read_csv(filepath)
print(f"数据形状: {df.shape}")
print(f"内存占用: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\n数据类型:")
print(df.dtypes)
print(f"\n前5行:")
df.head()

数据形状: (12256906, 5)
内存占用: 619.52 MB

数据类型:
time               str
user_id          int64
item_id          int64
item_category    int64
behavior_type    int64
dtype: object

前5行:


,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


本项目数据集包含约 1226 万条用户行为记录。考虑到数据规模较大，直接使用 Pandas 全量加载可能产生较高的内存开销，因此采用 Dask 进行分块读取和计算，并结合数据类型优化。最终转换为Parquet存储提升数据处理效率。

In [66]:
ddf = dd.read_csv(
    filepath,
    blocksize='128MB'
)

print("数据分区数:", ddf.npartitions)
print(ddf)

数据分区数: 3
Dask DataFrame Structure:
                 time user_id item_id item_category behavior_type
npartitions=3                                                    
               string   int64   int64         int64         int64
                  ...     ...     ...           ...           ...
                  ...     ...     ...           ...           ...
                  ...     ...     ...           ...           ...
Dask Name: to_string_dtype, 2 expressions
Expr=ArrowStringConversion(frame=FromMapProjectable(3ea9b56))


In [67]:
display(ddf.head())

,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


### 2. 数据清洗

In [68]:
print(ddf.dtypes)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print("数值型字段:", numeric_cols)

time             string
user_id           int64
item_id           int64
item_category     int64
behavior_type     int64
dtype: object
数值型字段: ['user_id', 'item_id', 'item_category', 'behavior_type']


ID变量: 'user_id', 'item_id'
类别变量: 'item_category', 'behavior_type'
时间变量：'time'

数据不存在连续型数值变量，因此不适用基于IQR或3σ的连续变量异常值检测方法。针对本数据集，异常值识别主要依据业务规则和字段取值范围。
查看类别变量behavior_type取值范围：

In [69]:
ddf['behavior_type'].value_counts().compute()

behavior_type
3      343564
1    11550581
2      242556
4      120205
Name: count, dtype: int64

检查数据中是否有负值或0值：

In [70]:
# 检查负值、极端值
for col in numeric_cols:
    negative_count = (ddf[col] < 0).sum().compute()
    print(f"\n{col}: 负值数量 = {negative_count}")

for col in ['user_id', 'item_id', 'item_category']:
    zero_count = (ddf[col] == 0).sum().compute()
    print(f"{col}: 值为0的数量 = {zero_count}")


user_id: 负值数量 = 0

item_id: 负值数量 = 0

item_category: 负值数量 = 0

behavior_type: 负值数量 = 0
user_id: 值为0的数量 = 0
item_id: 值为0的数量 = 0
item_category: 值为0的数量 = 0


检查缺失值：

In [71]:
print("user_id 缺失数:", df['user_id'].isna().sum())
print("item_id 缺失数:", df['item_id'].isna().sum())
print("item_category 缺失数:", df['item_category'].isna().sum())
print("behavior_type 缺失数:", df['behavior_type'].isna().sum())
print("time 缺失数:", df['time'].isna().sum())

user_id 缺失数: 0
item_id 缺失数: 0
item_category 缺失数: 0
behavior_type 缺失数: 0
time 缺失数: 0


In [ ]:
检查时间变量：

In [72]:
print(ddf['time'].head(10))
print(ddf['time'].unique()[:20])

0    2025-12-06 02
1    2025-12-09 20
2    2025-12-18 11
3    2025-12-06 10
4    2025-12-16 21
5    2025-12-03 20
6    2025-12-13 20
7    2025-11-27 16
8    2025-12-11 23
9    2025-12-05 23
Name: time, dtype: string
Dask Series Structure:
npartitions=3
    string
       ...
       ...
       ...
Dask Name: try_loc, 5 expressions
Expr=LocUnknown(frame=Unique(frame=ArrowStringConversion(frame=FromMapProjectable(3ea9b56))['time']), iindexer=slice(None, 20, None))


基于用户-商品-行为-时间四元组去重：

In [73]:
dup_groups = ddf.groupby(['user_id', 'item_id', 'behavior_type', 'time']).size()
dup_groups = dup_groups[dup_groups > 1].reset_index().rename(columns={0: 'count'})
dup_groups = dup_groups.compute()

print(f"重复四元组数量: {len(dup_groups)}")
print("\n重复的四元组及出现次数:")
dup_groups

重复四元组数量: 3843197

重复的四元组及出现次数:


,user_id,item_id,behavior_type,time,count
0,4913,3007091,1,2025-11-27 14,3
1,4913,6848743,1,2025-11-18 21,3
2,4913,6848743,1,2025-12-09 10,2
3,4913,10450747,1,2025-12-13 18,3
4,4913,12265214,1,2025-11-24 10,5
...,...,...,...,...,...
3843192,142455899,359793277,1,2025-11-22 19,2
3843193,142455899,366768925,1,2025-11-23 10,2
3843194,142455899,369133772,1,2025-11-30 13,2
3843195,142455899,378223383,1,2025-11-22 18,2


删除重复，每条只保留一行：

In [74]:
before = ddf.shape[0].compute()
print(f"去重前行数: {before:,}")

ddf = ddf.drop_duplicates(subset=['user_id', 'item_id', 'behavior_type', 'time'], keep='first')
after = ddf.shape[0].compute()
print(f"去重后行数: {after:,}")
print(f"删除重复行: {before - after:,}")

去重前行数: 12,256,906
去重后行数: 6,213,379
删除重复行: 6,043,527


统一时间格式，补齐为 YYYY-MM-DD HH:00:00，并校验时间有效性

In [75]:
ddf['time'].dtype

<StringDtype(na_value=<NA>)>

In [76]:
# 统一时间格式：补齐为 YYYY-MM-DD HH:00:00
ddf['time'] = dd.to_datetime(ddf['time'], format='%Y-%m-%d %H', errors='coerce')

In [77]:
ddf['time'].head(10)

0    2025-12-06 02:00:00
2    2025-12-18 11:00:00
16   2025-12-14 12:00:00
22   2025-11-28 09:00:00
23   2025-12-05 11:00:00
24   2025-11-23 21:00:00
25   2025-11-29 09:00:00
28   2025-12-15 14:00:00
34   2025-12-04 21:00:00
35   2025-11-24 20:00:00
Name: time, dtype: datetime64[us]

检查是否有转换失败的：

In [78]:
ddf['time'].isna().sum().compute()

np.int64(0)

In [79]:
# 查看时间范围
print(f"\n时间范围: {ddf['time'].min().compute()} 到 {ddf['time'].max().compute()}")


时间范围: 2025-11-18 00:00:00 到 2025-12-18 23:00:00


In [80]:
ddf['time'].dtype

dtype('<M8[us]')

可以确认时间变量已经被转换为datetime64类型。

### 3. 自动化数据检验脚本

In [81]:
def data_quality_check(ddf, id_columns, category_columns, time_column):
    """
    自动化数据质量校验函数
    返回质量报告 DataFrame
    """
    report = {}
    report['检测时间'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    report['数据总行数'] = ddf.shape[0].compute()
    report['数据总列数'] = ddf.shape[1]
    
    #1. 完整性检测：缺失值 
    missing = ddf.isnull().sum().compute()
    missing_pct = (missing / report['数据总行数'] * 100).round(2)
    missing_df = pd.DataFrame({
        '缺失数量': missing,
        '缺失率%': missing_pct
    })
    missing_df = missing_df[missing_df['缺失数量'] > 0]
    report['完整性_缺失值检测'] = '通过' if len(missing_df) == 0 else f'发现 {len(missing_df)} 列有缺失'
    report['完整性_缺失明细'] = missing_df
    
    # 2. 一致性检测：数值范围 
    # 类别变量取值分布
    category_report = {}
    for col in category_columns:
        value_counts = ddf[col].value_counts().compute()
        category_report[col] = value_counts
    report['一致性_类别变量取值分布'] = category_report
    
    # ID变量零值/负值检查
    id_report = {}
    for col in id_columns:
        zero_count = (ddf[col] == 0).sum().compute()
        neg_count = (ddf[col] < 0).sum().compute()
        id_report[col] = {'零值数量': zero_count, '负值数量': neg_count}
    report['一致性_ID变量检查'] = id_report
    
    # 时间有效性检查
    invalid_time = ddf[time_column].isna().sum().compute()
    time_min = ddf[time_column].min().compute()
    time_max = ddf[time_column].max().compute()
    report['一致性_时间无效数量'] = invalid_time
    report['一致性_时间范围'] = f'{time_min} 至 {time_max}'
    
    # 3. 唯一性检测：重复值 
    key_columns = id_columns + category_columns + [time_column]
    dup_groups = ddf.groupby(key_columns).size()
    dup_groups = dup_groups[dup_groups > 1].compute()
    
    if len(dup_groups) > 0:
        redundant = (dup_groups - 1).sum()
        report['唯一性_重复组数'] = len(dup_groups)
        report['唯一性_应删除行数'] = redundant
        report['唯一性_检测结果'] = f'发现 {len(dup_groups):,} 组重复，应删除 {redundant:,} 行'
    else:
        report['唯一性_检测结果'] = '通过，无重复'
    
    return report


# 执行质量校验
quality_report = data_quality_check(
    ddf=ddf,
    id_columns=['user_id', 'item_id'],
    category_columns=['item_category', 'behavior_type'],
    time_column='time'
)

# 打印报告
print("=" * 50)
print("数据质量校验报告")
print("=" * 50)
for key, value in quality_report.items():
    if key not in ['完整性_缺失明细', '一致性_类别变量取值分布', '一致性_ID变量检查']:
        print(f"\n{key}: {value}")

print("\n\n完整性_缺失明细:")
print(quality_report['完整性_缺失明细'])

print("\n一致性_ID变量检查:")
print(pd.DataFrame(quality_report['一致性_ID变量检查']))

print("\n一致性_类别变量取值分布:")
for col, dist in quality_report['一致性_类别变量取值分布'].items():
    print(f"\n{col}:")
    print(dist)

数据质量校验报告

检测时间: 2026-09-07 00:07:37

数据总行数: 6213379

数据总列数: 5

完整性_缺失值检测: 通过

一致性_时间无效数量: 0

一致性_时间范围: 2025-11-18 00:00:00 至 2025-12-18 23:00:00

唯一性_检测结果: 通过，无重复


完整性_缺失明细:
Empty DataFrame
Columns: [缺失数量, 缺失率%]
Index: []

一致性_ID变量检查:
      user_id  item_id
零值数量        0        0
负值数量        0        0

一致性_类别变量取值分布:

item_category:
item_category
3          2
8        496
10       129
21        88
28        61
        ... 
14051     32
14056     15
14066    120
14074      4
14080     49
Name: count, Length: 8916, dtype: int64

behavior_type:
behavior_type
3     331350
1    5535879
2     239472
4     106678
Name: count, dtype: int64


### 4. 中间表构建

In [ ]:
构建用户､商品､时间三个维度的基础聚合中间表。
其中用户维度中间表主键为user_id，统计每个用户的整体行为，包括总行为次数、各类行为次数（浏览、点击、购买等）、活跃天数、最早/最晚行为时间。
商品维度中间表主键为item_id，统计每个商品的表现，包括行为总次数、各类行为次数、独立用户数(多少用户进行过操作)、首次/最后出现时间。
时间维度中间表按日期聚合，统计每天电商平台的整体情况，包括行为总次数、各类行为次数、活跃用户数，活跃商品数。

In [82]:
# 先提取日期和小时
ddf_with_time = ddf.assign(
    date=ddf['time'].dt.date,
    hour=ddf['time'].dt.hour
)

# 1. 用户维度中间表 
# 用户行为分类计数
user_behavior = ddf_with_time.groupby(['user_id', 'behavior_type']).agg(
    次数=('behavior_type', 'size')
).reset_index().compute()

user_pivot = user_behavior.pivot_table(
    index='user_id',
    columns='behavior_type',
    values='次数',
    fill_value=0
).reset_index()
user_pivot.columns = ['user_id', '浏览', '收藏', '加购', '购买']
user_pivot['浏览'] = user_pivot['浏览'].astype('int32')
user_pivot['收藏'] = user_pivot['收藏'].astype('int32')
user_pivot['加购'] = user_pivot['加购'].astype('int32')
user_pivot['购买'] = user_pivot['购买'].astype('int32')

# 用户行为总次数
user_total = ddf_with_time.groupby('user_id').agg(
    总行为次数=('behavior_type', 'size'),
    最早行为时间=('time', 'min'),
    最晚行为时间=('time', 'max')
).compute().reset_index()

# 活跃天数
user_active_days = ddf_with_time.groupby('user_id')['date'].nunique().compute().reset_index()
user_active_days.columns = ['user_id', '活跃天数']
user_active_days['活跃天数'] = user_active_days['活跃天数'].astype('int16')

# 合并
user_agg = user_total.merge(user_active_days, on='user_id', how='left')
user_agg = user_agg.merge(user_pivot, on='user_id', how='left')
user_agg['总行为次数'] = user_agg['总行为次数'].astype('int32')

print("用户维度中间表构建完成")
print(f"用户数量: {len(user_agg):,}")

# 2. 商品维度中间表 
# 商品行为分类计数
item_behavior = ddf_with_time.groupby(['item_id', 'behavior_type']).agg(
    次数=('behavior_type', 'size')
).reset_index().compute()

item_pivot = item_behavior.pivot_table(
    index='item_id',
    columns='behavior_type',
    values='次数',
    fill_value=0
).reset_index()
item_pivot.columns = ['item_id', '浏览', '收藏', '加购', '购买']
item_pivot['浏览'] = item_pivot['浏览'].astype('int32')
item_pivot['收藏'] = item_pivot['收藏'].astype('int32')
item_pivot['加购'] = item_pivot['加购'].astype('int32')
item_pivot['购买'] = item_pivot['购买'].astype('int32')

# 商品行为总次数
item_total = ddf_with_time.groupby('item_id').agg(
    总行为次数=('behavior_type', 'size'),
    最早出现时间=('time', 'min'),
    最晚出现时间=('time', 'max')
).compute().reset_index()

# 独立用户数
item_users = ddf_with_time.groupby('item_id')['user_id'].nunique().compute().reset_index()
item_users.columns = ['item_id', '独立用户数']
item_users['独立用户数'] = item_users['独立用户数'].astype('int32')

# 合并
item_agg = item_total.merge(item_users, on='item_id', how='left')
item_agg = item_agg.merge(item_pivot, on='item_id', how='left')
item_agg['总行为次数'] = item_agg['总行为次数'].astype('int32')

print("商品维度中间表构建完成")
print(f"商品数量: {len(item_agg):,}")

# 3. 时间维度中间表（按日期聚合）
# 每天的各类行为计数
time_behavior = ddf_with_time.groupby(['date', 'behavior_type']).agg(
    次数=('behavior_type', 'size')
).reset_index().compute()

time_pivot = time_behavior.pivot_table(
    index='date',
    columns='behavior_type',
    values='次数',
    fill_value=0
).reset_index()
time_pivot.columns = ['date', '浏览', '收藏', '加购', '购买']
time_pivot['浏览'] = time_pivot['浏览'].astype('int32')
time_pivot['收藏'] = time_pivot['收藏'].astype('int32')
time_pivot['加购'] = time_pivot['加购'].astype('int32')
time_pivot['购买'] = time_pivot['购买'].astype('int32')

# 每天总行为次数
time_total = ddf_with_time.groupby('date').agg(
    总行为次数=('behavior_type', 'size')
).compute().reset_index()

# 活跃用户数和商品数
time_users = ddf_with_time.groupby('date')['user_id'].nunique().compute().reset_index()
time_users.columns = ['date', '活跃用户数']
time_users['活跃用户数'] = time_users['活跃用户数'].astype('int32')

time_items = ddf_with_time.groupby('date')['item_id'].nunique().compute().reset_index()
time_items.columns = ['date', '活跃商品数']
time_items['活跃商品数'] = time_items['活跃商品数'].astype('int32')

# 合并
time_agg = time_total.merge(time_users, on='date', how='left')
time_agg = time_agg.merge(time_items, on='date', how='left')
time_agg = time_agg.merge(time_pivot, on='date', how='left')
time_agg['总行为次数'] = time_agg['总行为次数'].astype('int32')

print("时间维度中间表构建完成")
print(f"日期数量: {len(time_agg):,}")

# 查看三张表的数据类型
print("\n用户表类型:")
print(user_agg.dtypes)
print("\n商品表类型:")
print(item_agg.dtypes)
print("\n时间表类型:")
print(time_agg.dtypes)

用户维度中间表构建完成
用户数量: 10,000
商品维度中间表构建完成
商品数量: 2,876,947
时间维度中间表构建完成
日期数量: 31

用户表类型:
user_id             int64
总行为次数               int32
最早行为时间     datetime64[us]
最晚行为时间     datetime64[us]
活跃天数                int16
浏览                  int32
收藏                  int32
加购                  int32
购买                  int32
dtype: object

商品表类型:
item_id             int64
总行为次数               int32
最早出现时间     datetime64[us]
最晚出现时间     datetime64[us]
独立用户数               int32
浏览                  int32
收藏                  int32
加购                  int32
购买                  int32
dtype: object

时间表类型:
date     object
总行为次数     int32
活跃用户数     int32
活跃商品数     int32
浏览        int32
收藏        int32
加购        int32
购买        int32
dtype: object


检查dataframe：

In [83]:
user_agg.head()

,user_id,总行为次数,最早行为时间,最晚行为时间,活跃天数,浏览,收藏,加购,购买
0,98047837,951,2025-11-18 12:00:00,2025-12-18 13:00:00,30,888,28,28,7
1,98607707,710,2025-11-18 02:00:00,2025-12-18 21:00:00,31,663,15,11,21
2,104221274,2058,2025-11-18 19:00:00,2025-12-18 21:00:00,29,1980,57,10,11
3,101260672,85,2025-11-18 20:00:00,2025-12-18 15:00:00,17,79,0,3,3
4,104811265,82,2025-12-01 13:00:00,2025-12-15 20:00:00,7,58,11,3,10


In [84]:
item_agg.head()

,item_id,总行为次数,最早出现时间,最晚出现时间,独立用户数,浏览,收藏,加购,购买
0,232431562,5,2025-12-03 16:00:00,2025-12-07 23:00:00,1,4,0,0,1
1,64749712,1,2025-12-18 11:00:00,2025-12-18 11:00:00,1,1,0,0,0
2,262661866,1,2025-12-14 12:00:00,2025-12-14 12:00:00,1,1,0,0,0
3,395905225,26,2025-11-18 21:00:00,2025-12-18 15:00:00,17,23,0,2,1
4,328893812,7,2025-11-25 20:00:00,2025-12-05 11:00:00,2,4,1,1,1


时间表的 date 列还是 object，转成标准日期类型：

In [86]:
time_agg['date'] = pd.to_datetime(time_agg['date'])
print(time_agg.dtypes)

date     datetime64[s]
总行为次数            int32
活跃用户数            int32
活跃商品数            int32
浏览               int32
收藏               int32
加购               int32
购买               int32
dtype: object


In [87]:
time_agg.head()

,date,总行为次数,活跃用户数,活跃商品数,浏览,收藏,加购,购买
0,2025-12-06,198378,6440,150061,176579,8279,10614,2906
1,2025-12-18,190183,6582,145031,170008,7404,9604,3167
2,2025-12-14,204056,6668,155900,182926,8086,9954,3090
3,2025-11-28,172392,6189,132833,154289,6427,8800,2876
4,2025-12-05,183633,6367,139341,163581,7430,9739,2883


输出处理后标准化数据集(Parquet格式)

In [88]:
user_agg.to_parquet('user_agg.parquet', compression='snappy', index=False)
item_agg.to_parquet('item_agg.parquet', compression='snappy', index=False)
time_agg.to_parquet('time_agg.parquet', compression='snappy', index=False)

In [89]:
# 导出用户行为主表（清洗后的明细数据）
ddf.to_parquet(
    'user_behavior_main.parquet',
    compression='snappy',
    write_index=False
)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
